# 3D Structure Viewer

In [1]:
!pip install py3dmol -q 

In [2]:
import py3Dmol
import numpy as np
import dask.dataframe as dd
import pandas as pd
from utils.distances_database_manager import get_data_with_max_distance
from utils.pdb_parser import get_atom_coordinates_from_pdb, open_pdb
from utils.vector_operations import calculate_direction, scale_vector
from IPython.display import display, Markdown

In [3]:
# path
database_path = '../data_parquet_starpep/'
dataset_path = '../datasets/StarPep/StarPep.csv'
pdbs_path = '../datasets/StarPep/ESMFold_pdbs/'

In [4]:
# functions
def get_data(database_path, dataset_path, pdbs_path, distance_function='euclidean'):
    # Get the result from the database based on the selected distance function
    result = get_data_with_max_distance(database_path, distance_function)

    # Extract values from the result
    sequence = result['sequence'].iloc[0]
    euclidean_value = result['euclidean'].iloc[0]
    angular_separation_value = result['angular_separation'].iloc[0]
    residue_source = int(result['aminoacid_source'].iloc[0]) + 1
    residue_target = int(result['aminoacid_target'].iloc[0]) + 1

    # Get PDB file based on the sequence
    sequences = pd.read_csv(dataset_path)
    sequence_id = sequences.loc[sequences['sequence'] == sequence, 'id'].iloc[0]
    pdb_file = f"{pdbs_path + sequence_id}.pdb"

    # Get atom coordinates
    pdb_data = open_pdb(pdb_file)
    coordinates = get_atom_coordinates_from_pdb(pdb_data, 'CA')
    residue_source_coord = coordinates[residue_source - 1]
    residue_target_coord = coordinates[residue_target - 1]

    # Calculate direction for both residues
    residue_source_dir = calculate_direction(residue_source_coord)
    residue_target_dir = calculate_direction(residue_target_coord)

    # Return the values in a dictionary
    data = {
        'sequence': sequence,
        'sequence_length': len(sequence),
        'euclidean_value': euclidean_value,
        'angular_separation_value': angular_separation_value,
        'residue_source': residue_source,
        'residue_target': residue_target,
        'pdb_file': pdb_file,
        'residue_source_coord': residue_source_coord,
        'residue_target_coord': residue_target_coord,
        'residue_source_dir': residue_source_dir,
        'residue_target_dir': residue_target_dir
    }

    return data

## 3D structure with Maximum Euclidean distance

In [5]:
# get data with maximum Euclidean distance
data_eu = get_data(database_path, dataset_path, pdbs_path, distance_function='euclidean')

In [11]:
# display result
display(Markdown("<br>**Data with the maximum Euclidean distance:**<br>"))
display(Markdown(f"**Sequence:** {data_eu['sequence']}"))
display(Markdown(f"**Euclidean Distance:** {data_eu['euclidean_value']}"))
display(Markdown(f"**Angular Separation Distance:** {data_eu['angular_separation_value']}"))

# view structure
view = py3Dmol.view(width='1600px', height='1200px')
view.setBackgroundColor('white')
view.addModel(open(data_eu['pdb_file'], 'r').read(),'pdb')

view.setStyle({'cartoon': {'color':'spectrum', 'thickness':0.4, 'style':'rectangle'}})

view.rotate(90,'y',1);
view.zoom(0.055)

<br>**Data with the maximum Euclidean distance:**<br>

**Sequence:** RRLRPRRPRLPRPRPRPRPRPRSLPLPRPKPRPIPRPLPLPRPRPKPIPRPLPLPRPRPRRIPRPLPLPRPRPRPIPRPLPLPQPQPSPIPRPL

**Euclidean Distance:** 254.05627925772916

**Angular Separation Distance:** 0.05805035375471346

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [12]:
# view structure
view = py3Dmol.view(width='1600px', height='1200px')
view.setBackgroundColor('white')
view.addModel(open(data_eu['pdb_file'], 'r').read(),'pdb')

view.addStyle({'and':[{'model':0}]}, {'cartoon': {'opacity':0.8,'color':'white'}})
view.addStyle({'and':[{'model':0},{'chain':'A'}]}, {'stick': {'radius': 0.2, 'color':'grey', 'dashedBonds': False, 'singleBonds': False}})
view.addStyle({'and':[{'model':0},{'chain':'A'}]}, {'sphere': {'radius': 0.3, 'color':'grey'}})
view.addStyle({'and':[{'model':0},{'atom':'CA'}]}, {'sphere': {'radius': 0.6, 'colorscheme':'blueCarbon'}})
view.addStyle({'and':[{'model':0},{'resi':data_eu['residue_source'], 'atom':'CA'}]}, {'sphere': {'radius': 0.8, 'colorscheme':'redCarbon'}})
view.addStyle({'and':[{'model':0},{'resi':data_eu['residue_target'], 'atom':'CA'}]}, {'sphere': {'radius': 0.8, 'colorscheme':'redCarbon'}})

view.addCylinder({
    "start":{'resi': data_eu['residue_source'], 'atom': 'CA'},
    "end":{'resi': data_eu['residue_target'], 'atom': 'CA'},
    "radius":0.1,
    "fromCap":1,
    "toCap":1,
    "color":"red",
    'dashed': True,
    'dashLength': 0.5,
    'gapLength': 0.8 
})

view.rotate(90,'y',1);
view.zoom(0.055)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 3D structure with Maximum Angular Separation Distance

In [8]:
# get data with maximum Angular Separation distance
data_as = get_data(database_path, dataset_path, pdbs_path, distance_function='angular_separation')

In [9]:
# display result
display(Markdown(f"**Sequence:** {data_as['sequence']}"))
display(Markdown(f"**Sequence Length:** {data_as['sequence_length']}"))
display(Markdown(f"**Euclidean Distance:** {data_as['euclidean_value']}"))
display(Markdown(f"**Angular Separation Distance:** {data_as['angular_separation_value']}"))

# view structure
view = py3Dmol.view(width='1600px', height='1200px')
view.setBackgroundColor('white')
view.addModel(open(data_as['pdb_file'], 'r').read(),'pdb')

view.setStyle({'cartoon': {'color':'spectrum', 'thickness':0.4, 'style':'rectangle'}})

view.rotate(90,'y',1);
view.zoom(0.055)

**Sequence:** MWHLKLFAVLMICLLLLAQVDGSPIPQQSSAKRRPRRMTPFWRAVSLRPIGASCRDDSECITRLCRKRRCSLSVAQE

**Sequence Length:** 77

**Euclidean Distance:** 100.83061655876004

**Angular Separation Distance:** 0.9999999670461953

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [13]:
# view structure
view = py3Dmol.view(width='1600px', height='1200px')
view.setBackgroundColor('white')
view.addModel(open(data_as['pdb_file'], 'r').read(),'pdb')

view.addStyle({'and':[{'model':0}]}, {'cartoon': {'opacity':0.8,'color':'white'}})
view.addStyle({'and':[{'model':0},{'chain':'A'}]}, {'stick': {'radius': 0.2, 'color':'grey', 'dashedBonds': False, 'singleBonds': False}})
view.addStyle({'and':[{'model':0},{'chain':'A'}]}, {'sphere': {'radius': 0.3, 'color':'grey'}})
view.addStyle({'and':[{'model':0},{'atom':'CA'}]}, {'sphere': {'radius': 0.6, 'colorscheme':'blueCarbon'}})
view.addStyle({'and':[{'model':0},{'resi':data_as['residue_source'], 'atom':'CA'}]}, {'sphere': {'radius': 0.8, 'colorscheme':'redCarbon'}})
view.addStyle({'and':[{'model':0},{'resi':data_as['residue_target'], 'atom':'CA'}]}, {'sphere': {'radius': 0.8, 'colorscheme':'redCarbon'}})

view.addCylinder({
    "start":{'resi': data_as['residue_source'], 'atom': 'CA'},
    "end":{'resi': data_as['residue_target'], 'atom': 'CA'},
    "radius":0.1,
    "fromCap":1,
    "toCap":1,
    "color":"red",
    'dashed': True,
    'dashLength': 0.5,
    'gapLength': 0.8 
})


# Scale the direction vectors
residue_source_dir = scale_vector(data_as['residue_source_dir'], 50)

view.addArrow({
    "start": {'resi':data_as['residue_source'], 'atom':'CA'},  
    "end": {'x': residue_source_dir[0], 'y': residue_source_dir[1], 'z': residue_source_dir[2]},
    "radius": 0.2,
    "color": "yellow"    
})

# Scale the direction vectors
residue_target_dir = scale_vector(data_as['residue_target_dir'], 30)
 
view.addArrow({
    "start": {'resi':data_as['residue_target'], 'atom':'CA'}, 
    "end": {'x': residue_target_dir[0], 'y': residue_target_dir[1], 'z': residue_target_dir[2]},
    "radius": 0.2,
    "color": "yellow"
})

view.rotate(90,'y');
#view.show()
view.zoom(0.1)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
view.png()